In [25]:
import pandas as pd
import numpy as np
import folium

from sklearn.cluster import DBSCAN

In [26]:
df = pd.read_csv("../data/clean_chicago_crimes.csv")

print(df.shape)
df.head()

(1388164, 20)


,id,date,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,year,latitude,longitude,hour,day_of_week,month,month_name,season,year_month
0,25699,2021-01-01 03:48:00,HOMICIDE,FIRST DEGREE MURDER,AUTO,False,False,224,2,3.0,38.0,2021,41.808891,-87.616577,3,Friday,1,January,Winter,2021-01
1,25700,2021-01-01 18:06:00,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,1532,15,28.0,25.0,2021,41.885721,-87.747018,18,Friday,1,January,Winter,2021-01
2,25701,2021-01-02 21:55:00,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,1022,10,24.0,29.0,2021,41.858214,-87.704772,21,Saturday,1,January,Winter,2021-01
3,25702,2021-01-03 20:09:00,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,332,3,5.0,43.0,2021,41.771062,-87.586271,20,Sunday,1,January,Winter,2021-01
4,25703,2021-01-03 21:28:00,HOMICIDE,FIRST DEGREE MURDER,APARTMENT,True,False,424,4,10.0,46.0,2021,41.736660,-87.544454,21,Sunday,1,January,Winter,2021-01


In [27]:
geo = df[['latitude', 'longitude']].dropna().copy()

geo = geo[
    geo['latitude'].between(-90, 90) &
    geo['longitude'].between(-180, 180)
]

print("Valid geographic records:", len(geo))

geo.head()


Valid geographic records: 1367657


,latitude,longitude
0,41.808891,-87.616577
1,41.885721,-87.747018
2,41.858214,-87.704772
3,41.771062,-87.586271
4,41.736660,-87.544454


Geographic aggregation
Nearby coordinates are rounded to reduce duplicate and near-duplicate geographic observations.

The number of crimes represented by each aggregated location is stored in crime_count.

In [28]:
geo['latitude_round'] = geo['latitude'].round(3)
geo['longitude_round'] = geo['longitude'].round(3)

geo_agg = (
    geo
    .groupby(
        ['latitude_round', 'longitude_round'],
        sort=False
    )
    .size()
    .reset_index(name='crime_count')
)

geo_agg = geo_agg.rename(columns={
    'latitude_round': 'latitude',
    'longitude_round': 'longitude'
})

print("Original crime records:", len(geo))
print("Aggregated locations:", len(geo_agg))

geo_agg.head()

Original crime records: 1367657
Aggregated locations: 42119


,latitude,longitude,crime_count
0,41.809,-87.617,426
1,41.886,-87.747,156
2,41.858,-87.705,56
3,41.771,-87.586,123
4,41.737,-87.544,106


In [29]:
coords_rad = np.radians(
    geo_agg[['latitude', 'longitude']].values
)

print("Coordinate matrix:", coords_rad.shape)

Coordinate matrix: (42119, 2)


DBSCAN configuration
A 100-meter radius is used to identify local micro-hotspots.

Crime counts from aggregated locations are used as sample weights.

In [30]:
earth_radius_km = 6371.0088

eps_km = 0.1
min_samples = 20

epsilon = eps_km / earth_radius_km

dbscan = DBSCAN(
    eps=epsilon,
    min_samples=min_samples,
    metric='haversine',
    algorithm='ball_tree',
    n_jobs=-1
)

geo_agg['cluster'] = dbscan.fit_predict(
    coords_rad,
    sample_weight=geo_agg['crime_count']
)

In [31]:
n_clusters = len(
    set(geo_agg['cluster']) - {-1}
)

noise_locations = (
    geo_agg['cluster'] == -1
).sum()

noise_crimes = geo_agg.loc[
    geo_agg['cluster'] == -1,
    'crime_count'
].sum()

print("Clusters detected:", n_clusters)
print("Noise locations:", noise_locations)
print("Crime records represented as noise:", noise_crimes)


Clusters detected: 5480
Noise locations: 6474
Crime records represented as noise: 26067


Build hotspot dataset
Noise is excluded.

Each DBSCAN cluster is summarized using:

weighted hotspot latitude
weighted hotspot longitude
total crime count
number of aggregated locations

In [32]:
clustered = geo_agg[
    geo_agg['cluster'] != -1
].copy()

hotspots = (
    clustered
    .groupby('cluster')
    .apply(
        lambda g: pd.Series({
            'latitude': np.average(
                g['latitude'],
                weights=g['crime_count']
            ),
            'longitude': np.average(
                g['longitude'],
                weights=g['crime_count']
            ),
            'crime_count': g['crime_count'].sum(),
            'locations': len(g)
        }),
        include_groups=False
    )
    .reset_index()
)

hotspots.head()

,cluster,latitude,longitude,crime_count,locations
0,0,41.809,-87.621964,3128.0,41.0
1,1,41.886,-87.760232,2157.0,34.0
2,2,41.858,-87.708243,526.0,8.0
3,3,41.771,-87.584168,748.0,4.0
4,4,41.737,-87.543788,132.0,3.0


In [33]:
hotspots = (
    hotspots
    .sort_values(
        'crime_count',
        ascending=False
    )
    .reset_index(drop=True)
)

hotspots['rank'] = hotspots.index + 1

hotspots = hotspots[
    [
        'rank',
        'cluster',
        'latitude',
        'longitude',
        'crime_count',
        'locations'
    ]
]

hotspots.head(20)

,rank,cluster,latitude,longitude,crime_count,locations
0,1,313,41.882,-87.635465,8479.0,49.0
1,2,21,41.910,-87.721177,7366.0,87.0
2,3,91,41.751,-87.614237,7075.0,74.0
3,4,1120,41.884,-87.634854,7035.0,47.0
4,5,362,41.893,-87.627134,6651.0,29.0
5,6,1426,41.891,-87.637540,5699.0,78.0
6,7,309,41.896,-87.689223,5574.0,72.0
7,8,11,41.779,-87.690896,5060.0,76.0
8,9,237,41.780,-87.629050,4495.0,71.0
9,10,143,41.866,-87.715309,4477.0,48.0


In [34]:
print("Total detected hotspots:", len(hotspots))

print(
    "Total crimes inside hotspots:",
    int(hotspots['crime_count'].sum())
)

print(
    "Largest hotspot crime count:",
    int(hotspots['crime_count'].max())
)

Total detected hotspots: 5480
Total crimes inside hotspots: 1341590
Largest hotspot crime count: 8479


Top crime hotspots
The complete DBSCAN result can contain thousands of local clusters.

For visualization, only the highest-ranked hotspots are displayed.

In [35]:
top_n = 50

top_hotspots = hotspots.head(top_n).copy()

top_hotspots.head(20)

,rank,cluster,latitude,longitude,crime_count,locations
0,1,313,41.882,-87.635465,8479.0,49.0
1,2,21,41.910,-87.721177,7366.0,87.0
2,3,91,41.751,-87.614237,7075.0,74.0
3,4,1120,41.884,-87.634854,7035.0,47.0
4,5,362,41.893,-87.627134,6651.0,29.0
5,6,1426,41.891,-87.637540,5699.0,78.0
6,7,309,41.896,-87.689223,5574.0,72.0
7,8,11,41.779,-87.690896,5060.0,76.0
8,9,237,41.780,-87.629050,4495.0,71.0
9,10,143,41.866,-87.715309,4477.0,48.0


In [36]:
center_lat = geo['latitude'].mean()
center_lon = geo['longitude'].mean()

hotspot_map = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles="OpenStreetMap"
)

In [37]:
max_crimes = top_hotspots['crime_count'].max()

for _, row in top_hotspots.iterrows():

    radius = 6 + (
        row['crime_count'] / max_crimes
    ) * 18

    folium.CircleMarker(
        location=[
            row['latitude'],
            row['longitude']
        ],
        radius=radius,
        color='red',
        weight=2,
        fill=True,
        fill_color='red',
        fill_opacity=0.65,
        popup=folium.Popup(
            f"""
            <b>Hotspot Rank:</b> {int(row['rank'])}<br>
            <b>Cluster:</b> {int(row['cluster'])}<br>
            <b>Crimes:</b> {int(row['crime_count'])}<br>
            <b>Locations:</b> {int(row['locations'])}
            """,
            max_width=250
        )
    ).add_to(hotspot_map)


In [38]:
hotspot_map

In [39]:
hotspots.to_csv(
    "../data/crime_hotspots.csv",
    index=False
)

print("crime_hotspots.csv saved ✅")


crime_hotspots.csv saved ✅


In [40]:
top_hotspots.to_csv(
    "../data/top_50_crime_hotspots.csv",
    index=False
)

print("top_50_crime_hotspots.csv saved ✅")

top_50_crime_hotspots.csv saved ✅


In [41]:
hotspot_map.save(
    "crime_hotspots_dbscan.html"
)

print("Hotspot map saved ✅")

Hotspot map saved ✅


Conclusion
DBSCAN was used to detect local crime concentrations from the complete geographic dataset.

The original crime records were spatially aggregated to reduce computational complexity while preserving crime frequency through sample weights.

A 100-meter DBSCAN neighborhood was selected as the operational definition of a micro-hotspot for this analysis.

The detected clusters were ranked by total crime count.

Outputs:

crime_hotspots.csv contains all detected hotspots.
top_50_crime_hotspots.csv contains the 50 highest-ranked hotspots.
crime_hotspots_dbscan.html provides an interactive visualization of the top hotspots.